# Yesi — vidéos gratuites sur Google Colab

1. Menu **Exécution → Modifier le type d'exécution → GPU T4** (gratuit).
2. Mets la photo de Yesi et ta vidéo dans **Mon Drive → dossier `yesi`** (appli Google Drive).
3. Lance les cellules **une par une** dans l'ordre (bouton ▶ de chaque cellule).

**Partie A — LivePortrait** : reprend les expressions, mouvements de tête et de lèvres d'une vidéo face caméra (play-back, parler, réagir).
**Partie B — MimicMotion** : reprend une danse en pied.

⚠️ N'utilise que des vidéos filmées par toi ou par une personne qui a donné son accord. Le son n'est pas repris : ajoute la musique dans TikTok, et active l'étiquette « Contenu généré par l'IA ».

In [ ]:
#@title 0. Vérifier le GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Partie A — LivePortrait (visage, play-back)

In [ ]:
#@title A1. Installer LivePortrait (~3 min)
%cd /content
!git clone -q --depth 1 https://github.com/KwaiVGI/LivePortrait
%cd /content/LivePortrait
!pip -q install tyro pykalman lmdb ffmpeg-python onnx onnxruntime-gpu albumentations scikit-image imageio imageio-ffmpeg rich
from huggingface_hub import snapshot_download
snapshot_download('KwaiVGI/LivePortrait', local_dir='/content/LivePortrait/pretrained_weights', allow_patterns=['liveportrait/*','insightface/*'])

In [ ]:
#@title A2. Prendre la photo et la vidéo dans Google Drive (dossier « yesi »)
from google.colab import drive
drive.mount('/content/drive')
import glob, os
d='/content/drive/MyDrive/yesi'
fs=glob.glob(d+'/*')
imgs=[f for f in fs if f.lower().endswith(('.png','.jpg','.jpeg','.heic','.webp'))]
vids=[f for f in fs if f.lower().endswith(('.mp4','.mov','.m4v'))]
assert imgs, 'Aucune photo dans Mon Drive/yesi'
assert vids, 'Aucune vidéo dans Mon Drive/yesi'
img, vid = imgs[0], vids[0]
print('Photo :', img); print('Vidéo :', vid)

In [ ]:
#@title A3. Générer (la vidéo est enregistrée dans Mon Drive/yesi)
!python inference.py -s "{img}" -d "{vid}" -o /content/out_lp --flag_crop_driving_video
import glob, shutil
out=[f for f in glob.glob('/content/out_lp/*.mp4') if 'concat' not in f]
dst='/content/drive/MyDrive/yesi/yesi_resultat.mp4'; shutil.copy(out[0], dst)
print('Terminé ! Ouvre l’appli Google Drive → dossier yesi → yesi_resultat.mp4')

## Partie B — MimicMotion (danse en pied)

Avant : connecte-toi sur huggingface.co et clique **Agree** sur https://huggingface.co/stabilityai/stable-video-diffusion-img2vid-xt-1-1 (modèle de base, gratuit). Crée un token *Read* dans Settings → Access Tokens.

In [ ]:
#@title B1. Connexion Hugging Face (colle ton token dans la case)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
#@title B2. Installer MimicMotion (~5 min)
%cd /content
!git clone -q --depth 1 https://github.com/Tencent/MimicMotion
%cd /content/MimicMotion
!pip -q install diffusers==0.27.0 huggingface_hub==0.25.2 einops omegaconf decord av onnxruntime-gpu matplotlib
!mkdir -p models/DWPose
from huggingface_hub import hf_hub_download
for f in ['yolox_l.onnx','dw-ll_ucoco_384.onnx']: hf_hub_download('yzd-v/DWPose', f, local_dir='models/DWPose')
hf_hub_download('tencent/MimicMotion','MimicMotion_1-1.pth', local_dir='models')

In [ ]:
#@title B3. Envoyer la photo de Yesi (en pied) puis la vidéo de danse (plan fixe, en pied)
from google.colab import files
print('1) Photo de Yesi en pied'); dimg=list(files.upload().keys())[0]
print('2) Vidéo de danse'); dvid=list(files.upload().keys())[0]

In [ ]:
#@title B4. Générer (T4 : ~10-20 min pour ~3 s) et télécharger
num_frames = 16 #@param {type:"integer"}
resolution = 576 #@param {type:"integer"}
cfg=f'''base_model_path: stabilityai/stable-video-diffusion-img2vid-xt-1-1
ckpt_path: models/MimicMotion_1-1.pth
test_case:
  - ref_video_path: /content/MimicMotion/{dvid}
    ref_image_path: /content/MimicMotion/{dimg}
    num_frames: {num_frames}
    resolution: {resolution}
    frames_overlap: 4
    num_inference_steps: 25
    noise_aug_strength: 0
    guidance_scale: 2.0
    sample_stride: 2
    fps: 15
    seed: 42
'''
open('configs/yesi.yaml','w').write(cfg)
!python inference.py --inference_config configs/yesi.yaml --output_dir /content/out_mm
import glob; from google.colab import files
out=sorted(glob.glob('/content/out_mm/*.mp4')); print(out); files.download(out[-1])

### Si « CUDA out of memory »
Baisse `num_frames` à 12 ou `resolution` à 512, puis relance B4.